# Constrained static portfolio optimisation

This notebook demonstrates the public API with reproducible synthetic inputs. It compares a maximum-Sharpe allocation with a minimum-variance allocation under position and asset-bucket constraints.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from portfolio_optimizer import (
    PortfolioReporter,
    StaticOptimizer,
    create_style_constraints,
)

## Synthetic inputs

The example uses generated expected returns and a positive-semidefinite covariance matrix. No market or proprietary data is included.

In [ ]:
rng = np.random.default_rng(42)
assets = [f"Asset {letter}" for letter in "ABCDEFGH"]
expected_returns = rng.uniform(0.02, 0.15, len(assets))
factor = rng.normal(size=(len(assets), len(assets)))
covariance = factor @ factor.T * 0.01
risk_free_rate = 0.02
growth_assets = assets[:3]

## Solve two allocations

Each position is capped at 40%, while the first three assets must jointly represent between 30% and 50% of the portfolio. The optimiser checks solver success and constraint residuals before returning.

In [ ]:
optimizer = StaticOptimizer(expected_returns, covariance)
bounds = tuple((0.0, 0.40) for _ in assets)
constraints = create_style_constraints(
    assets,
    growth_assets,
    lower=0.30,
    upper=0.50,
)

max_sharpe = optimizer.maximize_sharpe(
    risk_free_rate=risk_free_rate,
    bounds=bounds,
    constraints=constraints,
)
minimum_variance = optimizer.minimize_variance(
    bounds=bounds,
    constraints=constraints,
)

print(f"Maximum constraint violation: {max_sharpe.max_constraint_violation:.2e}")

## Summarise and visualise

In [ ]:
portfolios = {
    "Maximum Sharpe": max_sharpe.x,
    "Minimum variance": minimum_variance.x,
}
reporter = PortfolioReporter(assets, risk_free_rate=risk_free_rate)
summary = reporter.create_summary_table(portfolios, expected_returns, optimizer.covariance)
display(reporter.format_performance_table(summary))

weights = pd.DataFrame(portfolios, index=assets)
reporter.plot_weights_grouped(weights, title="Constrained portfolio weights")
plt.show()